### Morphological disparity: sparse landmarks vs NSM latents

Do lizard families and life-history groups rank the same way for morphological
disparity whether shape is measured from 28 sparse landmarks or from NSM latent
codes? 

Procrustes variance is summed over dimensions, so the two spaces aren't
comparable on raw values. Ranks within each space are the comparable unit, and
the latents are also reduced to 84 PCs to confirm the ordering isn't an artifact
of dimensionality.

* load and check the landmark configurations (centroid size, consensus vs atlas mean)
* identify the coordinate axes and split symmetric from asymmetric shape
* disparity per family and per trait in each representation → plot.

### Setup and paths

In [ ]:
# Imports and paths
import os, re, json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from NSM.plotting import load_mrk_json, dumbbell_plot
from NSM.morphometrics import (centroid_size, dist_to_mean, morphol_disparity,
                               mshape, procrustes_dist, two_d_array, lm_diff, check_axis_labels)

# Specify training directory and atlas directory
RUN          = "run_v72"                         # training attempt directory
ATLAS_RUN    = "2026_07-15_13_06_22/"           # atlas/builder run that produced alignedLMs
DROPBOX_ROOT = Path("/home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/")
SPECIES_CSV = "../lizard_species_list.csv"

# Build other directories relative to those above
cwd      = Path.cwd()
base_wd  = cwd.parent
train_dir = base_wd / RUN
os.chdir(train_dir)
print(f"Working directory: {os.getcwd()}")

LM_DIR      = DROPBOX_ROOT / ATLAS_RUN / "alignedLMs"
ATLAS_DIR   = DROPBOX_ROOT / ATLAS_RUN / "atlas"
MEAN_LMS_FN = ATLAS_DIR / "atlas_sparse_landmarks.mrk.json"
CKPT        = "2500"

OUT_DIR = Path("gmm_results_lms_v_lats")
OUT_DIR.mkdir(exist_ok=True)
print(f"Outputs will be written to: {OUT_DIR.resolve()}")

# Load config and parse species / vertebra from filenames 
config_path = "model_params_config.json"
with open(config_path) as f:
    cfg = json.load(f)
print(f"\033[92mLoaded config from {config_path}\033[0m")

# Parse filenames
all_vtk_files = [os.path.basename(f) for f in cfg["list_mesh_paths"]]

# Load NSM latent codes
CKPT_PATH = f"latent_codes/{CKPT}.pth"
latent_ckpt = torch.load(CKPT_PATH, map_location="cpu")
codes = latent_ckpt["latent_codes"]["weight"].detach().cpu().numpy()
print(f"Latent codes: {codes.shape}")
assert len(codes) == len(all_vtk_files), (
    f"{len(codes)} latent codes but {len(all_vtk_files)} meshes in config -- order/count mismatch")

In [ ]:
# Build specimen metadata table -- species, family, life history, vertebral region

# Parse specimen ID and vertebra from filenames
pat = re.compile(r"^(?P<species>[\w\s\-]+)[\-_ ]+[\w\d]+[\-_ ]+(?P<vertebra>[CTL]?\d+)", re.IGNORECASE)
parsed = [pat.match(f) for f in all_vtk_files]
specimens = pd.DataFrame({"mesh":        all_vtk_files,
                          "specimen_id": [m.group("species") if m else None for m in parsed],
                          "vertebra":    [m.group("vertebra").upper() if m else None for m in parsed]})
print(f"Parsed {specimens['specimen_id'].notna().sum()} / {len(specimens)} filenames")

# Join against the species master list
sdf = pd.read_csv(SPECIES_CSV)
sdf["marker"] = sdf["marker"].astype(str).str.strip().str.strip("'\"")
specimens = specimens.merge(sdf, left_on="specimen_id", right_on="specimen", how="left")

unmatched = specimens.loc[specimens["family"].isna(), "specimen_id"].dropna().unique()
print(f"{specimens['family'].notna().sum()} / {len(specimens)} specimens matched to {SPECIES_CSV}")
if len(unmatched):
    print(f"\033[33mUnmatched specimen IDs ({len(unmatched)}):\033[0m {sorted(unmatched)[:10]}")
print("Specimens dataframe head:\n", specimens.head())

# One color per trait, derived from that trait's marker
marker_to_color = {'P': (0.65, 0.69, 0.12),   # pea soup
                   '+': (0.84, 0.65, 0.23),   # saffron
                   's': (0.72, 0.44, 0.22),   # mud
                   'd': (0.36, 0.557, 0.68),  # powder blue
                   'X': (0.10, 0.51, 0.40),   # deep blue
                   'o': (0.60, 0.50, 0.46),   # slate
                   '2': (0, 0, 0)}            # black

trait_marker = specimens.drop_duplicates("trait").set_index("trait")["marker"]
trait_colors = {t: marker_to_color.get(m, (0.5, 0.5, 0.5))
                for t, m in trait_marker.items() if pd.notna(t)}
unmapped = [t for t, m in trait_marker.items() if pd.notna(t) and m not in marker_to_color]
if unmapped:
    print(f"\033[33mTraits using a marker not in marker_to_color (defaulting to grey): {unmapped}\033[0m")
print(f"{len(trait_colors)} traits: {sorted(trait_colors)}")

# ── Family → trait composition, for striped colouring ────────────────────────
family_comp = (specimens.dropna(subset=["family", "trait"]).groupby("family")["trait"].value_counts(normalize=True).unstack(fill_value=0))
family_comp = family_comp[[t for t in trait_colors if t in family_comp.columns]]
family_trait = family_comp.idxmax(axis=1).to_dict()    # dominant trait, for sorting

### Load landmarks (`.mrk.json`) and build the shape array

In [ ]:
# Load 3D Slicer Atlas aligned and scaled landmark data
lm_coords = []
for fpath in all_vtk_files:
    lm_name = os.path.splitext(fpath)[0] + ".mrk.json"
    lm_path = LM_DIR / lm_name
    coords, _ = load_mrk_json(lm_path)
    lm_coords.append(coords)

lm_coords_3d = np.stack(lm_coords)   # (N, p, 3)
print(f"Landmark data shape - 3d: {lm_coords_3d.shape}")

# Atlas mean sparse landmarks (same landmark set as LM_DIR, from the setup cell)
mean_lms_3d, _ = load_mrk_json(MEAN_LMS_FN)
print(f"Atlas mean landmarks shape: {mean_lms_3d.shape}")

In [ ]:
# Data checks before analysis

# Are the configurations already centred and scaled?
cs = np.array([centroid_size(X) for X in lm_coords_3d])
print(f"Centroid size: mean={cs.mean():.5f}  sd={cs.std():.5f}  CV={100*cs.std()/cs.mean():.2f}%  "
      f"range=({cs.min():.5f}, {cs.max():.5f})")
if 100 * cs.std() / cs.mean() > 5:
    print("\033[33mCentroid size varies by >5% — the configurations may not be fully scaled. "
          "Consider dividing each by its centroid size before proceeding.\033[0m")

consensus = mshape(lm_coords_3d)
print(f"\nConsensus vs atlas mean landmarks: Procrustes distance = "
      f"{procrustes_dist(consensus, mean_lms_3d):.6f}")
print(f"Mean per-landmark offset = {np.linalg.norm(consensus - mean_lms_3d, axis=1).mean():.6f}")

d_mean = dist_to_mean(lm_coords_3d)
print(f"\nProcrustes distance to consensus: mean={d_mean.mean():.5f}  "
      f"median={np.median(d_mean):.5f}  max={d_mean.max():.5f}")

# Optional: rescale to unit centroid size (set to True if the check above complained)
RESCALE_TO_UNIT_CS = False
if RESCALE_TO_UNIT_CS:
    lm_coords_3d = np.stack([(X - X.mean(axis=0)) / centroid_size(X) for X in lm_coords_3d])
    consensus = mshape(lm_coords_3d)
    print("\nRescaled all configurations to unit centroid size.")

In [ ]:
# Identify which coordinate index is x (the axis of bilateral symmetry), y, and z

midline_idx = [20, 21, 8, 9, 12, 15]                                   # TO DO: midline landmarks (0-based)
paired_landmarks = {0: 5, 1: 4, 2: 3, 18: 19, 22: 23, 24: 25,
                    6: 7, 26: 27, 13: 14, 10: 11, 16: 17}              # TO DO: left: right pairs (0-based)
COTYLE_IDX, NEURAL_SPINE_IDX = 15, 12                                  # TO DO: for the y-axis check
ZYG_PR_IDX, ZYG_PO_IDX = 16, 0                                         # TO DO: for the z-axis check

# 1st check - axis of symmetry: midline landmarks are near-constant along x
x_idx = check_axis_labels("1st Check", "x", np.var(lm_coords_3d[:, midline_idx, :].reshape(-1, 3), axis=0), "min", 
                          f"Midline Landmarks {midline_idx}")

# 2nd check - the same axis should carry most of the left-right variation
lr_diffs = np.concatenate([lm_diff(lm_coords_3d, l, r) for l, r in paired_landmarks.items()], axis=0)
check_axis_labels("2nd Check", "x", np.var(lr_diffs, axis=0), "max",
                  f"Paired left-right landmarks (mirrored across X) {paired_landmarks}")

# 3rd check - cotyle vs neural spine separate along y
y_idx = check_axis_labels("3rd Check", "y", np.mean(np.abs(lm_diff(lm_coords_3d, NEURAL_SPINE_IDX, COTYLE_IDX)), axis=0), "max",
                          f"Cotyle vs neural spine landmarks ({COTYLE_IDX} & {NEURAL_SPINE_IDX})")

# 4th check - pre- vs post-zygapophyses separate along z
z_idx = check_axis_labels("4th Check", "z", np.mean(np.abs(lm_diff(lm_coords_3d, ZYG_PO_IDX, ZYG_PR_IDX)), axis=0), "max",
                          f"Pre- and post-zygapophyses landmarks ({ZYG_PR_IDX} & {ZYG_PO_IDX})")

# Build axis list using determined order
axis_order = [None, None, None]
axis_order[z_idx], axis_order[x_idx], axis_order[y_idx] = 'z', 'x', 'y'
print("\nAxis order:", axis_order)

In [ ]:
# Symmetric vs asymmetric
mirrored = lm_coords_3d.copy()
mirrored[:, :, x_idx] *= -1
for l, r in paired_landmarks.items():
    mirrored[:, [l, r], :] = mirrored[:, [r, l], :]

symm_component = (lm_coords_3d + mirrored) / 2.0
asym_component = (lm_coords_3d - mirrored) / 2.0
da_component   = asym_component.mean(axis=0, keepdims=True)     # directional asymmetry
fa_component   = asym_component - da_component                  # fluctuating asymmetry

# Variance ratios relative to symmetric component
sym_var = np.var(symm_component)
da_var  = np.var(da_component)
fa_var  = np.var(fa_component)

da_ratio = da_var / sym_var
fa_ratio = fa_var / sym_var

print(f"DA / symmetric variance ratio: {da_ratio:.4f}")
print(f"FA / symmetric variance ratio: {fa_ratio:.4f}")

# Decision threshold — analogous to checking p-values in geomorph ANOVA
THRESHOLD = 0.05  # i.e., asymmetry explains < 5% of individual shape variation

if fa_ratio < THRESHOLD and da_ratio < THRESHOLD:
    print("Asymmetry signal is negligible — proceed with symmetric component only")
    coords_analysis = symm_component
else:
    print("Asymmetry signal is meaningful — retain both components")
    coords_analysis = lm_coords_3d 

### Check morphological disparity (based on morphol_disparity in R geomorph)

In [ ]:
# ── Disparity: latents vs landmarks, with dimensionality robustness check ─────
# Procrustes variance is summed over dimensions, so the 512-dim latent space
# and 84-dim landmark configuration aren't comparable on raw values. Ranks
# within each space are the comparable unit — this cell confirms the family
# rank order is stable when latent dimensionality is matched to landmarks.

ITER    = 999
N_MATCH = 84          # landmark dimensionality (28 LMs × 3 coords)
MIN_N   = 5           # disparity is unreliable below this

# ── Representations ───────────────────────────────────────────────────────────
L_std = (codes - codes.mean(0)) / (codes.std(0) + 1e-12)
Y_lm  = two_d_array(coords_analysis)

# PCA-reduced latents at matched dimensionality
pca_L = PCA(n_components=N_MATCH, svd_solver="full").fit(L_std)
L_84    = pca_L.transform(L_std)
var_exp = pca_L.explained_variance_ratio_.sum()

print(f"Landmarks:      {Y_lm.shape}")
print(f"Latents (full): {L_std.shape}")
print(f"Latents ({N_MATCH} PCs): {L_84.shape}  ({100*var_exp:.1f}% of latent variance)")

REPS = [("Landmarks",            Y_lm),
        ("Latents_full",         L_std),
        (f"Latents_{N_MATCH}PC", L_84)]

# ── Disparity per group in each representation ───────────────────────────────
for col in ["family", "trait"]:
    keep = specimens[col].notna().values

    # Drop groups with too few specimens
    counts = specimens.loc[keep, col].value_counts()
    small  = counts[counts < MIN_N].index.tolist()
    if small:
        print(f"  Dropping n < {MIN_N}: {[(g, int(counts[g])) for g in small]}")
        keep &= ~specimens[col].isin(small).values

    groups_sub = specimens.loc[keep, col].values

    comp = pd.DataFrame()
    for rep_name, Y in REPS:
        var_tab, _, _ = morphol_disparity(Y[keep], groups_sub, iter=ITER)
        comp[f"var_{rep_name}"]  = var_tab
        comp[f"rank_{rep_name}"] = var_tab.rank(ascending=False).astype(int)

    # Drop singleton groups — zero variance is an artifact, not a finding
    singletons = comp[comp.filter(like="var_").sum(axis=1) == 0].index.tolist()
    if singletons:
        print(f"\n  Dropping singleton groups (zero variance): {singletons}")
        comp = comp.drop(index=singletons)
        for rep_name, _ in REPS:
            comp[f"rank_{rep_name}"] = comp[f"var_{rep_name}"].rank(ascending=False).astype(int)

    comp["shift_vs_LM"] = comp["rank_Landmarks"] - comp[f"rank_Latents_{N_MATCH}PC"]
    comp = comp.sort_values("rank_Landmarks")

    # Key check: do full and reduced latents agree on the ordering?
    rho_dim, p_dim = spearmanr(comp["rank_Latents_full"],
                               comp[f"rank_Latents_{N_MATCH}PC"])
    rho_lm,  p_lm  = spearmanr(comp["rank_Landmarks"],
                               comp[f"rank_Latents_{N_MATCH}PC"])

    print(f"\n{'='*70}")
    print(f"  Disparity by {col}  (n groups = {len(comp)})")
    print(f"{'='*70}")
    print(f"  Full vs {N_MATCH}-PC latents:  Spearman r = {rho_dim:+.3f}, p = {p_dim:.4f}")
    print(f"    → r > 0.9 means the rank order is robust to dimensionality")
    print(f"  Landmarks vs {N_MATCH}-PC latents: Spearman r = {rho_lm:+.3f}, p = {p_lm:.4f}")

    show = ["rank_Landmarks", "rank_Latents_full",
            f"rank_Latents_{N_MATCH}PC", "shift_vs_LM"]
    print(f"\n{comp[show].to_string()}")

    # Families the two representations disagree about most
    big = comp[comp["shift_vs_LM"].abs() >= 8].sort_values("shift_vs_LM", ascending=False)
    if len(big):
        print(f"\n  Largest disagreements (|shift| ≥ 8):")
        for grp, r in big.iterrows():
            direction = "MORE" if r["shift_vs_LM"] > 0 else "less"
            print(f"    {grp:<22s} {direction:>4s} disparate in latent space  "
                  f"(LM {int(r['rank_Landmarks']):>2d} → "
                  f"latent {int(r[f'rank_Latents_{N_MATCH}PC']):>2d})")

    if col == "family":
        comp_family = comp.copy()

    comp.to_csv(OUT_DIR / f"disparity_dimcheck_{col}.csv")

print(f"\nSaved → {OUT_DIR.resolve()}")

# ── Pooled by life history: mean family rank per trait ───────────────────────
fam2trait = (specimens.dropna(subset=["family", "trait"])
                      .groupby("family")["trait"]
                      .agg(lambda s: s.value_counts().idxmax()))
comp_family["trait"] = comp_family.index.map(fam2trait)

rank_cols = [f"rank_{name}" for name, _ in REPS]
pooled = comp_family.groupby("trait")[rank_cols].mean()
pooled["n_families"] = comp_family.groupby("trait").size()
pooled = pooled.sort_values("rank_Landmarks")

print(f"\n{'='*70}")
print("  Pooled by life history (mean family disparity rank)")
print(f"{'='*70}")
print(pooled.to_string(float_format=lambda v: f"{v:6.1f}"))

pooled.to_csv(OUT_DIR / "disparity_pooled_by_trait.csv")

In [ ]:
# ── Dumbbell plots: disparity rank shift, landmarks vs NSM latents ────────────
N_MATCH     = 84
RANK_COL    = f"rank_Latents_{N_MATCH}PC"
MIN_SHIFT   = 8       # groups below this are muted and unlabelled
GREY        = (0.62, 0.62, 0.62)
PURE_THRESH = 0.80    # families above this are drawn as one solid colour
MIN_FRAC    = 0.05    # traits below this are left out of a family's stripes

# Family colors
family_colors   = {f: trait_colors.get(t, GREY) for f, t in family_trait.items()}
family_segments = {}
for f, comp in family_comp.iterrows():
    if comp.max() >= PURE_THRESH:
        family_segments[f] = [(1.0, family_colors[f])]
    else:
        cols = [trait_colors.get(t, GREY) for t, frac in comp.items() if frac >= MIN_FRAC]
        family_segments[f] = [(1.0 / len(cols), c) for c in cols]

In [ ]:
# ── Family: striped by trait composition, small shifts muted ─────────────────
# Row order (trait block, then rank within block) is set here, not inside dumbbell().
fam_df = pd.read_csv(OUT_DIR / "disparity_dimcheck_family.csv", index_col=0)
trait_order = {t: i for i, t in enumerate(trait_colors)}
fam_df["_trait"]  = [family_trait.get(g) for g in fam_df.index]
fam_df["_torder"] = fam_df["_trait"].map(trait_order).fillna(len(trait_order))
fam_df = fam_df.sort_values(["_torder", "rank_Landmarks"])

dumbbell_plot(fam_df,
                OUT_DIR / "disparity_dumbbell_family.png",
                ref_col="rank_Landmarks",
                cmp_col=RANK_COL,
                colors=family_colors,
                segments=family_segments,
                bands=fam_df["_trait"],
                band_colors=trait_colors,
                min_shift=MIN_SHIFT,
                grey=GREY,
                right_pad=4.5,
                figsize=(9, 11))

In [ ]:
# ── Specimen-level life history summary ──────────────────────────────────────
fam = pd.read_csv(OUT_DIR / "disparity_dimcheck_family.csv", index_col=0)

spec = specimens.dropna(subset=["family", "trait"])[["family", "trait"]].copy()
spec["rank_Landmarks"] = spec["family"].map(fam["rank_Landmarks"])
spec[RANK_COL]         = spec["family"].map(fam[RANK_COL])
spec = spec.dropna(subset=["rank_Landmarks", RANK_COL])
spec = spec[spec["trait"] != "snake"]

tr = (spec.groupby("trait")[["rank_Landmarks", RANK_COL]].mean()
          .reindex([t for t in trait_colors if t in set(spec["trait"])]))
tr.index.name = "trait"
tr.to_csv(OUT_DIR / "disparity_shift_by_trait_specimen.csv")
print(tr.to_string(float_format=lambda v: f"{v:6.2f}"))
print(spec["trait"].value_counts().to_string())

dumbbell_plot(tr.sort_values("rank_Landmarks"),
                OUT_DIR / "disparity_dumbbell_shift_by_trait.png",
                ref_col="rank_Landmarks",
                cmp_col=RANK_COL,
                colors=trait_colors,
                min_shift=None,
                grey=GREY,
                right_pad=1.0,
                figsize=(7.5, 3.6))